# Sistema de Gestão Pecuária — Análise Reprodutiva
**TCC - Ciência da Computação**

* **Autor:** Gabriel de Oliveira Irineu
* **Matrícula:** 1230113289

---

## Objetivos

### Geral
Criar um pipeline de análise de dados para gestão reprodutiva bovina unificando Python, SQL e Power BI.

### Específicos
- Processar dados brutos e carregar no SQLite.
- Calcular o Intervalo Entre Partos (IEP) por matriz.
- Mapear a sazonalidade dos partos ao longo do ano.
- Exportar base tratada para dashboard no Power BI.

---

## Pipeline de Execução
1. Configuração de diretórios
2. Carga e limpeza de dados
3. Modelagem relacional em SQL
4. Análise de IEP e produtividade
5. Análise de sazonalidade
6. Exportação para Power BI

## Configuração de Diretórios

Define os caminhos usados no projeto e cria as pastas necessárias caso ainda não existam.

In [ ]:
from pathlib import Path
import sqlite3
import matplotlib.pyplot as plt
import pandas as pd

# Configuração visual dos gráficos
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "ggplot")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["font.size"] = 10

# Mapeamento de pastas
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SQL_DIR = ROOT_DIR / "sql"
REPORTS_DIR = ROOT_DIR / "reports"

# Criação das pastas
for folder in [RAW_DIR, PROCESSED_DIR, SQL_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Diretório raiz: {ROOT_DIR}")
print("Pastas configuradas com sucesso.")

## Carga e Limpeza de Dados

Lê as três abas da planilha de origem (`Matrizes`, `Touros` e `Eventos_Partos`), padroniza os nomes das colunas-chave, ajusta os tipos numéricos e converte as colunas de data.

Registros de parto sem `ID_Matriz` preenchido são mantidos na base bruta, mas excluídos das análises seguintes.

In [ ]:
caminho_planilha = RAW_DIR / "Gados.xlsx"

try:
    with pd.ExcelFile(caminho_planilha) as xls:
        df_matrizes = pd.read_excel(xls, sheet_name="Matrizes")
        df_touros = pd.read_excel(xls, sheet_name="Touros")
        df_partos = pd.read_excel(xls, sheet_name="Eventos_Partos")

    # Padronização de chaves
    if "ID_Vaca" in df_matrizes.columns:
        df_matrizes.rename(columns={"ID_Vaca": "ID_Matriz"}, inplace=True)

    # Tipagem numérica
    df_partos["ID_Matriz"] = pd.to_numeric(df_partos["ID_Matriz"], errors="coerce").astype("Int64")
    df_partos["ID_Parto"] = pd.to_numeric(df_partos["ID_Parto"], errors="coerce").astype("Int64")

    # Conversão de datas
    df_partos["Data_Parto"] = pd.to_datetime(df_partos["Data_Parto"], format="%d/%m/%Y", errors="coerce")
    df_matrizes["Data_Nascimento"] = pd.to_datetime(df_matrizes["Data_Nascimento"], errors="coerce")

    # Limpeza de texto
    df_partos["Sexo_Cria"] = df_partos["Sexo_Cria"].str.upper().str.strip()

    print(f"Matrizes: {len(df_matrizes)}")
    print(f"Touros: {len(df_touros)}")
    print(f"Partos: {len(df_partos)}")
    print(f"Partos sem ID_Matriz: {df_partos['ID_Matriz'].isna().sum()}")

    display(df_matrizes.head(3))
    display(df_partos.head(3))

except FileNotFoundError:
    print(f"Arquivo não encontrado em: {caminho_planilha}")

## Modelagem Relacional em SQL

Carrega as tabelas tratadas em um banco SQLite local. A query de validação confirma quantos partos possuem uma matriz associada e quantos ficaram sem vínculo.

In [ ]:
caminho_banco = SQL_DIR / "fazenda.db"
conn = sqlite3.connect(caminho_banco)

try:
    df_matrizes_sql = df_matrizes.copy()
    df_partos_sql = df_partos.copy()

    df_matrizes_sql["Data_Nascimento"] = df_matrizes_sql["Data_Nascimento"].dt.strftime("%Y-%m-%d")
    df_partos_sql["Data_Parto"] = df_partos_sql["Data_Parto"].dt.strftime("%Y-%m-%d")

    df_matrizes_sql.to_sql("matrizes", conn, if_exists="replace", index=False)
    df_touros.to_sql("touros", conn, if_exists="replace", index=False)
    df_partos_sql.to_sql("eventos_partos", conn, if_exists="replace", index=False)

    print("Banco de dados SQLite atualizado.")

    query_val = """
    SELECT
        COUNT(CASE WHEN ID_Matriz IS NOT NULL THEN 1 END) AS partos_validos,
        COUNT(CASE WHEN ID_Matriz IS NULL THEN 1 END) AS partos_orfaos
    FROM eventos_partos;
    """
    display(pd.read_sql(query_val, conn))

finally:
    conn.close()

## Ranking de Matrizes e Distribuição por Sexo

Conta o número de partos por matriz e calcula a proporção de crias machos e fêmeas registradas.

In [ ]:
conn = sqlite3.connect(caminho_banco)

try:
    # Ranking de partos por matriz
    query_ranking = """
    SELECT
        m.ID_Matriz,
        m.Nome AS Nome_Vaca,
        COUNT(p.ID_Parto) AS Total_Partos
    FROM matrizes m
    INNER JOIN eventos_partos p ON m.ID_Matriz = p.ID_Matriz
    GROUP BY m.ID_Matriz, m.Nome
    ORDER BY Total_Partos DESC;
    """
    df_ranking = pd.read_sql(query_ranking, conn)

    # Distribuição por sexo da cria
    query_sexo = """
    SELECT
        COALESCE(Sexo_Cria, 'Não Informado') AS Sexo_Cria,
        COUNT(*) AS Total,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM eventos_partos WHERE ID_Matriz IS NOT NULL), 2) AS Porcentagem
    FROM eventos_partos
    WHERE ID_Matriz IS NOT NULL
    GROUP BY Sexo_Cria;
    """
    df_sexo = pd.read_sql(query_sexo, conn)

    print("Ranking por matriz:")
    display(df_ranking.head())

    print("\nProporção por sexo:")
    display(df_sexo)

finally:
    conn.close()

## Cálculo do Intervalo Entre Partos (IEP)

Usa uma window function para comparar a data de cada parto com a data do parto anterior da mesma matriz, gerando o IEP em dias e em meses. Matrizes com apenas um parto registrado não entram nesse cálculo, pois não há um intervalo anterior para comparar.

In [ ]:
conn = sqlite3.connect(caminho_banco)

try:
    query_iep = """
    WITH PartosOrdenados AS (
        SELECT
            p.ID_Parto,
            p.ID_Matriz,
            m.Nome AS Nome_Vaca,
            p.Data_Parto,
            LAG(p.Data_Parto) OVER (PARTITION BY p.ID_Matriz ORDER BY p.Data_Parto) AS Data_Parto_Anterior
        FROM eventos_partos p
        INNER JOIN matrizes m ON p.ID_Matriz = m.ID_Matriz
    )
    SELECT
        ID_Matriz,
        Nome_Vaca,
        Data_Parto_Anterior,
        Data_Parto AS Data_Parto_Atual,
        CAST((julianday(Data_Parto) - julianday(Data_Parto_Anterior)) AS INTEGER) AS IEP_Dias,
        ROUND((julianday(Data_Parto) - julianday(Data_Parto_Anterior)) / 30.4375, 1) AS IEP_Meses
    FROM PartosOrdenados
    WHERE Data_Parto_Anterior IS NOT NULL
    ORDER BY IEP_Dias ASC;
    """

    df_iep = pd.read_sql(query_iep, conn)
    display(df_iep)

    if not df_iep.empty:
        print(f"Média do IEP: {df_iep['IEP_Meses'].mean():.1f} meses")
        print(f"Mediana do IEP: {df_iep['IEP_Meses'].median():.1f} meses")

finally:
    conn.close()

## Sazonalidade dos Partos

Agrupa os partos válidos por mês para verificar se existe concentração de nascimentos em algum período do ano.

In [ ]:
df_partos_validos = df_partos.dropna(subset=["ID_Matriz"]).copy()

df_partos_validos["Mes_Num"] = df_partos_validos["Data_Parto"].dt.month

meses_map = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}
df_partos_validos["Mes_Nome"] = df_partos_validos["Mes_Num"].map(meses_map)

df_sazonalidade = (
    df_partos_validos
    .groupby(["Mes_Num", "Mes_Nome"])
    .size()
    .reset_index(name="Total_Partos")
    .sort_values(by="Mes_Num")
)

display(df_sazonalidade)

## Visualização dos Resultados

Consolida os quatro indicadores calculados (sexo, ranking, sazonalidade e IEP) em um painel único, salvo como imagem para consulta posterior.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Sexo da cria
axes[0, 0].pie(
    df_sexo["Total"],
    labels=df_sexo["Sexo_Cria"],
    autopct="%1.1f%%",
    colors=["#66B2FF", "#FF9999"]
)
axes[0, 0].set_title("Nascimentos por Sexo")

# Ranking matrizes
axes[0, 1].bar(df_ranking["Nome_Vaca"], df_ranking["Total_Partos"], color="#2ca02c")
axes[0, 1].set_title("Partos por Matriz")
axes[0, 1].tick_params(axis="x", rotation=45)

# Sazonalidade
axes[1, 0].plot(df_sazonalidade["Mes_Nome"], df_sazonalidade["Total_Partos"], marker="o", color="#ff7f0e")
axes[1, 0].set_title("Sazonalidade dos Partos")
axes[1, 0].tick_params(axis="x", rotation=45)

# IEP
if not df_iep.empty:
    axes[1, 1].bar(df_iep["Nome_Vaca"], df_iep["IEP_Meses"], color="#9467bd")
    axes[1, 1].set_title("IEP em Meses")
    axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
caminho_img = REPORTS_DIR / "dashboard_reproducao.png"
plt.savefig(caminho_img, dpi=300)
print(f"Imagem salva em: {caminho_img}")
plt.show()

## Exportação para Análise Externa

Exporta as tabelas tratadas e os indicadores calculados para uma planilha Excel, usada como fonte de dados para o dashboard no Power BI.

In [ ]:
caminho_excel_out = REPORTS_DIR / "Relatorio_Fazenda_Final_BR.xlsx"

with pd.ExcelWriter(caminho_excel_out, engine="openpyxl") as writer:
    df_matrizes.to_excel(writer, sheet_name="Dim_Matrizes", index=False)
    df_touros.to_excel(writer, sheet_name="Dim_Touros", index=False)
    df_partos.to_excel(writer, sheet_name="Fact_Partos", index=False)
    df_iep.to_excel(writer, sheet_name="Ind_IEP", index=False)
    df_sazonalidade.to_excel(writer, sheet_name="Ind_Sazonalidade", index=False)

print(f"Arquivo gerado em: {caminho_excel_out}")

## Limitações

A tabela `Touros` é carregada e disponibilizada no banco de dados, mas não existe um campo que relacione cada parto ao touro responsável. Por esse motivo, não foi possível cruzar informações de paternidade com os indicadores reprodutivos nesta versão.

O cálculo do IEP também é limitado pelo volume de dados: poucas matrizes possuem mais de um parto registrado, o que reduz o tamanho da amostra usada nesse indicador.